In [1]:
import numpy as np # массивы
import k3d # визуализауия
import matplotlib.pyplot as plt # графики
from collections import deque # для хранения результатов симуляций
from pykdtree.kdtree import KDTree # библиотека с готовым алгоритмом поиска расстояний между точками в двух больших массивах
import time

In [2]:
# Объявляем константы

vector3 = np.ndarray

cm = 1
m = 100*cm

golden_ratio_minus_1 = 2/(1+np.sqrt(5)) # Распределение фотоумножители сделано по спирали Фибоначчи, поэтому нам нужно вот такое число 

keV = 1e-3
MeV = 1

#### Как определить, попал ли фотон с координатами $(x_0, y_0, z_0) = \vec{r_0}$ и вектором направления $(n_x, n_y, n_z) = \vec{n}$ в ФЭУ?

Известно, что параметризация прямой при заданных $\vec{r_0}, \vec{n}$ выглядит как $\vec{r} = \vec{r_0}+\vec{n}t$
Необходимо найти t при котором $r^2=R$:

$x^2+y^2+z^2-R^2 = (x_0+n_xt)^2+(y_0+n_yt)^2+(z_0+z_xt)^2-R^2 = (n_x^2+n_y^2+n_z^2)t^2+2(x_0n_x+y_0n_y+z_0n_z)t-R^2+y_0^2+x_0^2+z_0^2=t^2+2(\vec{r_0}, \vec{n})t-R^2+r_0^2=0$

Решая квадратное уравнение, получаем:
$t=-(\vec{r_0}, \vec{n})+\sqrt{(\vec{r_0}, \vec{n})^2+R^2-r_0^2}$

Подставив, полученное t в параметризацию, найдём координаты пересечения траектории фотона со сферой. Далее можно применить функцию, определяющую принадлежит ли точка на поверхности ФЭУ

In [3]:
# Функция таймера из лекций

def timer(func):
    """Замеряет время выполнения функции."""
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__}: {elapsed:.4f} с")
        return result
    return wrapper

In [4]:
class Detector:

    def __init__(self, R : float, N : int, r : float, a : float): # При инициализации сразу создаём массив координат фотоумножителей
    
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.Lambda = 8*m

        # Если расположить центр фотокатода на поверхности сферы, то остальная его часть будет за её пределами
        # Считаем, на какое расстояние нужно отступить, чтобы фотокатод был погружен внутрь сферы

        beta = (np.pi-np.asin(r/R))/2
        delta_h = r/np.tan(beta)
        R_photo = R-delta_h

        # Алгоритм по распределению дететкоров по сфере взят из https://arxiv.org/pdf/0912.4540

        N_points = (N-1)/2

        if (N_points % 1 == 0):

            indices = np.arange(-N_points, N_points+1)
            
        else:

            indices = np.arange(np.floor(-N_points), N_points)

        # Изначально задаются сферические углы (тета сдвинут на пи пополам, чтобы покрыть верхнюю и нижнюю полусферы)
        
        theta = np.asin(2*indices/N)+np.pi/2
        phi = 2*np.pi*indices*golden_ratio_minus_1

        # Задаётся массив декартовых координат фотокатодов

        self.photo_coords = R_photo*np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)]).T

    # Функция по определнию того, сколько фотонов прилетело на каждый фотоумножитель
    @timer
    def is_PMT(self, x : float=0, y : float=0, z : float=0, vector: vector3=None) -> np.ndarray: 
        
        if vector is None:
            vec = np.array([[x, y, z]])
        else:
            vec = np.array(vector)

        DetectorHits = np.zeros(self.N, dtype=int) # Массив количества фотонов, пойманных каждым ФЭУ. Изначально нудевой
        # distances = np.linalg.norm(vec, axis=-1)

        # Для определения принадлежности точки к ФЭУ используется k-d-дерево
        
        tree = KDTree(self.photo_coords)
        neighbor_dists, neighbor_indices = tree.query(vec) # получаем массив расстояний до ближайшего ФЭУ и массив индексов этих фЭУ
        
        filt = (neighbor_dists <= self.r) # отбираем только точки, лежащие ближе 10 см к ФЭУ
        good_inds = neighbor_indices[filt]

        inds, counts = np.unique(good_inds, return_counts=True) # Сколько раз каждое число встречается в итоговом массиве

        DetectorHits[inds] += counts

        return DetectorHits # i-ый элемент полученного массива равен числу фотонов, попавших на i-ый ФЭУ

    # Функция, определяет конечные точки траекторий фотонов без учёта затухания. Суть метода изложена выше
    def GeneratePhotonEndPoints(self, source_coords : vector3, directions : vector3):
        
        source_coord_norm2 = np.linalg.norm(source_coords)**2
        scalar_product = (source_coords*directions).sum(axis=-1)
        Discr = scalar_product**2+self.R**2-source_coord_norm2
        roots = -scalar_product+np.sqrt(Discr)

        EndPoints = source_coords + directions*roots[:, np.newaxis]

        return EndPoints
    
    # Функция, определяет конечные точки траекторий фотонов с учётом затухания. Суть метода изложена выше
    def GeneratePhotonEndPointsWithAttenuation(self, source_coords : vector3, directions : vector3):

        N_photons = directions.shape[0]
        
        source_coord_norm2 = np.linalg.norm(source_coords)**2
        scalar_product = (source_coords*directions).sum(axis=-1)
        Discr = scalar_product**2+self.R**2-source_coord_norm2
        roots = -scalar_product+np.sqrt(Discr)
        
        EndMinusStartVec = directions*roots[:, np.newaxis]
        Distances = np.linalg.norm(EndMinusStartVec, axis=-1)
        
        probabilities = np.exp(-Distances/self.Lambda)
        unifrom_probs = np.random.uniform(size=N_photons)
        filt = (probabilities > unifrom_probs)

        EndPoints = source_coords + EndMinusStartVec[filt]

        return EndPoints

    # Метод, возвращающий число фотоэлектронов отдельно на каждом из N ФЭУ
    # Настоящий метод принадлежит классу Detector, поскольку фотоэлектроны образуются внутри ФЭУ.
    
    def GetPoissonCounts(self, hits):
        
        PhotoelectronNum = np.zeros(self.N)

        for i, n_hit in enumerate(hits):

            PhotoelectronNum[i] = np.random.poisson(self.a, size=n_hit).sum() # На каждом детекторе считается суммарное количество образованных фотоэлектронов

        return PhotoelectronNum

In [5]:
MyDet = Detector(15*m, 2000, 10*cm, 0.5) # Объявляем дететкор с заданными параметрами

In [6]:
positions = MyDet.photo_coords # положения ФЭУ
positions

array([[    0.        ,    -0.        ,  1499.9666663 ],
       [  -57.9283642 ,   -33.79134093,  1498.46669963],
       [   28.12004537,    90.5530974 ,  1496.96673296],
       ...,
       [   49.5075833 ,   105.01499938, -1495.4667663 ],
       [   28.12004537,   -90.5530974 , -1496.96673296],
       [  -57.9283642 ,    33.79134093, -1498.46669963]], shape=(2000, 3))

In [7]:
colours = np.full(positions.shape[0], 0xff0000)

In [8]:
# Визуализация геометрии. Синий шар - сцинтиллятор. Красные шарики - ФЭУ

plot = k3d.plot()
plot += k3d.points([0,0,0], point_size=2*MyDet.R)
plot += k3d.points(positions, point_size=2*MyDet.r, colors=colours)

plot.display()

C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\traittypes\traittypes.py:98: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [9]:
class Source: # Источник фотонов

    def __init__(self, Y : int):
    
        self.Y = Y

    def GeneratePhotonDirections(self, E): # изтропный генератор направлений от события с энергией E

        N_photons = int(self.Y*E) # Считаем, сколько фотонов должно образоваться

        # Изотропно -> равномерно по косинусу тета и равномерно по фи
        cos_theta = np.random.uniform(-1, 1, N_photons)
        sin_theta = np.sqrt(1-cos_theta**2)
        phi = np.random.uniform(0, 2*np.pi, N_photons)

        # Получаем единичные вектора направлений
        directions = np.array([sin_theta*np.cos(phi), sin_theta*np.sin(phi), cos_theta]).T
        
        return directions

In [10]:
MySource = Source(10000/MeV) # Объявляем источник

In [11]:
directions = MySource.GeneratePhotonDirections(10*MeV)

In [12]:
x0 = 0*cm
y0 = 0*cm
z0 = 0*cm

source_coordinates = np.array([x0, y0, z0]) # Положение источника в центре сферы

In [13]:
EndPoints = MyDet.GeneratePhotonEndPointsWithAttenuation(source_coordinates, directions)
Hits = MyDet.is_PMT(vector=EndPoints)
Hits

is_PMT: 0.0032 с


array([0, 0, 0, ..., 0, 0, 1], shape=(2000,))

In [14]:
Hits.sum()

np.int64(376)

### Работает быстро